# Notebook 00: Define Area of Interest

## 1 Connect Google Drive (Colab only)

This section is only required when running in **Google Colab** and your project/data are stored in Google Drive.

- Mounting Drive makes your repository and datasets accessible under `/content/drive`.
- If you run this notebook locally (Jupyter / VS Code), **skip this section**.

**Expected structure (Drive)**
After mounting, your project root should contain:
- `notebooks/`
- `src/`
- `config.yml` (or `config.yaml`)

Proceed to the code cell below to mount Drive.

In [ ]:
from google.colab import drive
import os

# Check if the drive is mounted
if os.path.exists('/content/drive'):
    # Try to unmount
    try:
        drive.flush_and_unmount()
        print("Successfully unmounted")
    except:
        print("Unmount failed, the drive might not be mounted or busy")

# Mount the drive
drive.mount('/content/drive')

Successfully unmounted
Mounted at /content/drive


**Troubleshooting:**  
- If Colab becomes disconnected, Reconnect the runtime and rerun the mounting cell.
- If we receive an error such as `Mountpoint must not already contain files`, delete all the sub-folders under "/content/drive" from the Files panel before retrying. We need to delete these one by one starting from the innermost folders, until the last "drive" folder is deleted.


## 2 Install packages (only if needed)

In most cases, **Google Colab already includes the packages required** for this workflow. The most common missing dependency is **`netCDF4`** (NetCDF I/O support).

### Check what is already installed (Colab)
Before installing anything, you can inspect the current environment by running `!pip list`.

- If all required packages are present and only `netCDF4` is missing, install **only `netCDF4`**.
- If other required packages are missing from `!pip list`, install them **together with** `netCDF4` in the code cell below.

### Local Jupyter note
If you are running in a **local environment** (Jupyter / VS Code), assume all dependencies were installed when preparing the environment following the **main repository README**. In that case, you can skip this section.

Proceed to the code cell below only when installation is necessary.

In [1]:
# In Google Colab, almost all packages already available, except netCDF4
!pip install netCDF4

Once this completes, you’ll be ready to load and manipulate spatial data (Sections 2.1 and beyond) and visualize your results on real-world map projections.

## 3 Project Folders

Next, we establish a clear folder structure on your mounted Drive so that raw inputs, downloads, intermediate masks, and final outputs each live in their own dedicated directory. The snippet below:

- Declares a single main project folder
- Builds subpaths for input data, downloaded files, mask outputs, and final results
- Ensures each required directory exists, creating it if necessary

A well-organized directory layout makes it easy to trace where every file came from and prevents accidental overwrites—setting you up for a reproducible, scalable workflow.


In [4]:
# Import library
import os, sys

# Configurable Directory
# Base folder on Google Drive for this project
# main_dir = '/content/drive/MyDrive/hybrid-bias-correction'
main_dir = "C:/Users/benny/OneDrive/Documents/Github/hybrid-bias-correction"

# Define key subfolders
input_dir     = f'{main_dir}/data/input'      # RAW input data goes here
downloads_dir = f'{main_dir}/data/downloads'  # All downloaded files
mask_dir      = f'{main_dir}/data/mask'       # Generated land masks
output_dir    = f'{main_dir}/data/output'     # Final outputs & results

# List of directories to ensure exist
for d in [input_dir, downloads_dir, mask_dir, output_dir]:
    os.makedirs(d, exist_ok=True)
    print(f"✅ Ensured directory exists: {d}")


✅ Ensured directory exists: C:/Users/benny/OneDrive/Documents/Github/hybrid-bias-correction/data/input
✅ Ensured directory exists: C:/Users/benny/OneDrive/Documents/Github/hybrid-bias-correction/data/downloads
✅ Ensured directory exists: C:/Users/benny/OneDrive/Documents/Github/hybrid-bias-correction/data/mask
✅ Ensured directory exists: C:/Users/benny/OneDrive/Documents/Github/hybrid-bias-correction/data/output


## 4 Area of Interest (AOI)

Accurate boundary delineation is essential for robust geospatial analysis. In our workflow, the IMERG grid—with its 0.1° spatial resolution—serves as the fundamental reference for all outputs. To maintain consistency between different datasets and ensure reliable spatial analysis, it is crucial to generate a global land mask that exactly conforms to the IMERG grid. This approach guarantees that all output data layers are aligned and comparable.

There are many global country (admin0) boundary datasets used by humanitarian and international development professionals. Some popular examples include:
- [Natural Earth](https://www.naturalearthdata.com/downloads/10m-cultural-vectors/10m-admin-0-countries/)
- [GADM](https://gadm.org/download_world.html)
- [geoBoundaries](https://www.geoboundaries.org/globalDownloads.html)
- [Fieldmaps.io](https://fieldmaps.io/data/adm0)

For this exercise, we will use the global admin0 boundary from Fieldmaps.io to develop a global land mask at 0.1° resolution.

Furthermore, the consistent global land mask is the key starting point for subsequent processes: users can easily clip this mask to their preferred Area of Interest (AOI), which is described in detail in Section 2.2.



### 4.1 Creating a Global Land Mask

Now that our workspace is organized, we’ll generate a world‐wide land mask perfectly aligned to the IMERG 0.1° grid. This mask will underpin **all** later AOI extractions and ensures that every subsequent raster operation uses the same spatial reference.

The code below:

1. **Downloads** a zipped GeoPackage of admin-0 boundaries from Fieldmaps.io  
2. **Extracts** the vector file into our downloads folder  
3. **Defines** a global 0.1° raster grid spanning latitude [–90, 90] and longitude [–180, 180]  
4. **Rasterizes** each country polygon to a mask where land = 1 and water = NaN  
5. **Builds** an xarray DataArray and **saves** it as a NetCDF for downstream use


In [ ]:
# Download and build global land mask
import os
import zipfile
import requests
import geopandas as gpd
import numpy as np
from rasterio import features
from affine import Affine
import xarray as xr

# Get the admin0 in GPKG format
def download_and_extract_gpkg(url: str, extract_dir: str, zip_fname: str = "adm0_polygons.gpkg.zip"):
    """
    Download a zipped GPKG from `url` into `zip_fname` and extract into `extract_dir`.
    """
    try:
        print(f"⏳ Downloading GPKG from {url} …")
        resp = requests.get(url, stream=True)
        resp.raise_for_status()

        # Write to local zip file
        with open(zip_fname, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"✔️ Downloaded to {zip_fname}")

        # Extract
        with zipfile.ZipFile(zip_fname, 'r') as z:
            z.extractall(extract_dir)
        print(f"✔️ Extracted GPKG to {extract_dir}")

    except requests.HTTPError as he:
        raise RuntimeError(f"HTTP error while downloading GPKG: {he}")
    except Exception as e:
        raise RuntimeError(f"Failed to download or extract GPKG: {e}")

# Rasterize the global land polygon
def rasterize_global_land(gpkg_path: str, out_nc: str, resolution: float = 0.1):
    """
    Read country boundaries from `gpkg_path`, rasterize to a global land mask at `resolution`,
    and save to NetCDF `out_nc`.
    """
    try:
        print(f"⏳ Reading country boundaries from {gpkg_path} …")
        gdf = gpd.read_file(gpkg_path)
        print(f"✔️ Loaded {len(gdf)} polygons")

        # Define global grid
        lon_min, lon_max = -180, 180
        lat_min, lat_max = -90, 90
        width = int((lon_max - lon_min) / resolution)
        height = int((lat_max - lat_min) / resolution)
        transform = Affine.translation(lon_min, lat_max) * Affine.scale(resolution, -resolution)
        print(f"✔️ Raster grid: {width}×{height} at {resolution}° resolution")

        # Rasterize: 1=land, NaN=water
        shapes = ((geom, 1) for geom in gdf.geometry)
        mask = features.rasterize(
            shapes,
            out_shape=(height, width),
            transform=transform,
            fill=np.nan,
            dtype="float32",
            all_touched=True
        )
        print("✔️ Rasterization complete")

        # Build xarray DataArray
        lon = lon_min + (np.arange(width) + 0.5) * resolution
        lat = lat_max - (np.arange(height) + 0.5) * resolution
        da = xr.DataArray(mask, coords={'lat': lat, 'lon': lon}, dims=['lat', 'lon'], name='land')
        da.attrs.update({
            "description": "Global land mask (1=land, NaN=water)",
            "grid_resolution": f"{resolution}°",
            "units": "1 / NaN"
        })

        # Save NetCDF
        print(f"⏳ Saving NetCDF to {out_nc} …")
        da.to_netcdf(out_nc)
        print(f"✅ World land mask generated at {out_nc}")

    except Exception as e:
        raise RuntimeError(f"Error during rasterization or saving NetCDF: {e}")

def main():
    try:
        bdir = os.path.join(main_dir, 'data/downloads/boundary')
        wdir = os.path.join(main_dir, 'data/mask/world')
        os.makedirs(bdir, exist_ok=True)
        os.makedirs(wdir, exist_ok=True)

        gpkg_url = "https://data.fieldmaps.io/adm0/osm/all/adm0_polygons.gpkg.zip"
        download_and_extract_gpkg(gpkg_url, bdir)

        gpkg = os.path.join(bdir, "adm0_polygons.gpkg")
        out_nc = os.path.join(wdir, "wld_land_mask.nc")
        rasterize_global_land(gpkg, out_nc)

        print("🎉 All done: global land mask is ready.")

    except Exception as e:
        print(f"❌ {e}")

# Run
if __name__ == "__main__":
    main()

With this NetCDF now in place, we can move on to Section 4.2 to clip it to any Area of Interest—whether by country code, bounding box, or our own custom shapefile.

### 4.2 AOI Options

Defining a precise Area of Interest (AOI) is crucial for targeted analysis. There are at least three options to obtain the AOI: (1) using publicly available administrative boundaries, (2) using a bounding box or predefined polygon, or (3) using our own data.

Using the global land mask created in Section 4.1, we can clip the world mask to any of these AOIs. The following subsections describe sample workflows for each approach, including scripts to extract the AOI from
- Global administrative boundaries (Section 4.2.1)
- Custom bounding boxes (Section 4.2.2)
- User-supplied datasets (Section 4.2.3).

For a smoother experience, we recommend using administrative boundaries from the first option, but we are free to choose whichever option best fits our needs!

By accurately isolating the focus area using one of these methods, we ensure that the data processing is relevant and efficient for specific requirements.

#### 4.2.1 Country Boundaries

In this section, we utilize publicly available administrative boundary datasets to select a desired region. This option is often preferred because it provides country boundaries, which are widely used by many humanitarian professionals.

Let’s first check the attributes of the country boundary dataset we have downloaded. This will help us identify the fields containing the ISO3 country code and the country name, which we will use to filter for specific countries.


In [5]:
# Inspect the GeoPackage attributes
import geopandas as gpd

# Path to your GPKG
gpkg_file = os.path.join(main_dir, 'data/downloads/boundary/adm0_polygons.gpkg')

# Read into GeoDataFrame
gdf = gpd.read_file(gpkg_file)

# Print the column names and first few rows
print("Available columns:", list(gdf.columns))
print(gdf.head())

ModuleNotFoundError: No module named 'geopandas'

After inspecting the attributes of the country boundaries, we’ll proceed to filter the data based on the **ISO3** codes `adm0_src` and **Country** names `adm0_name`.

For full list of ISO3 country code, we can refer to this page [https://en.wikipedia.org/wiki/ISO_3166-1_alpha-3](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-3).

Once the countries are selected, we can interactively preview them on a map to ensure that the boundaries are correct.

In [ ]:
# Interactive preview of selected polygons with Folium
import os
import geopandas as gpd
import folium
from folium.features import GeoJsonPopup

def load_and_filter_gpkg(gpkg_path: str, iso3_list: list):
    print(f"⏳ Loading GPKG from {gpkg_path} …")
    gdf = gpd.read_file(gpkg_path)
    print(f"✔️ Loaded {len(gdf)} features")

    print(f"⏳ Filtering for ISO3 codes: {iso3_list} …")
    gdf_f = gdf[gdf['adm0_src'].isin(iso3_list)].copy()
    print(f"✔️ {len(gdf_f)} features after filtering")

    # Drop any datetime columns (not JSON-serializable)
    dt_cols = gdf_f.select_dtypes(include=['datetime64']).columns
    if len(dt_cols):
        print(f"⏳ Dropping datetime columns: {list(dt_cols)} …")
        gdf_f.drop(columns=dt_cols, inplace=True)
        print("✔️ Datetime columns removed")

    return gdf_f

def build_folium_map(gdf, popup_fields: list, zoom_start: int = 4):
    # Compute map center from bounds
    minx, miny, maxx, maxy = gdf.total_bounds
    center = [(miny + maxy) / 2, (minx + maxx) / 2]
    print(f"⏳ Building Folium map centered at {center} …")

    m = folium.Map(location=center, zoom_start=zoom_start, tiles="CartoDB positron")
    folium.GeoJson(
        gdf,
        name="Filtered Polygons",
        popup=GeoJsonPopup(fields=popup_fields, labels=True),
        style_function=lambda feat: {
            "fillColor": "#228B22",
            "color": "black",
            "weight": 0.7,
            "fillOpacity": 0.3,
        }
    ).add_to(m)

    print("✅ Folium map is ready")
    return m

def main():
    try:
        gpkg = os.path.join(main_dir, 'data/downloads/boundary/adm0_polygons.gpkg')
        iso3_list = ['LKA', 'IND', 'BGD', 'PAK', 'NPL', 'BTN', 'MDV', 'AFG']
        popup_fields = ['adm0_src', 'adm0_name']

        gdf_filtered = load_and_filter_gpkg(gpkg, iso3_list)
        m = build_folium_map(gdf_filtered, popup_fields, zoom_start=4)

        # In Colab the last expression is displayed automatically:
        return m

    except Exception as e:
        print(f"❌ Error: {e}")

# Run
if __name__ == "__main__":
    main()

For this exercise, we will use **Sri Lanka** as selected country boundary, with the column `adm0_src` = 'LKA'

In [ ]:
# Generate and save AOI‐specific land mask
import os
import geopandas as gpd
import numpy as np
import xarray as xr
from rasterio import features
from affine import Affine

def load_world_mask(path: str) -> xr.DataArray:
    print(f"⏳ Loading world land mask from {path} …")
    da = xr.open_dataarray(path)
    print("✔️ World land mask loaded")
    return da

def filter_aoi(gpkg_path: str, iso3_code: str) -> gpd.GeoDataFrame:
    print(f"⏳ Loading GPKG from {gpkg_path} …")
    gdf = gpd.read_file(gpkg_path)
    print(f"✔️ Loaded {len(gdf)} features")

    print(f"⏳ Filtering for ISO3 == '{iso3_code}' …")
    gdf_aoi = gdf[gdf['adm0_src'] == iso3_code].copy()
    print(f"✔️ {len(gdf_aoi)} features after filtering")
    return gdf_aoi

def rasterize_and_clip(aoi_gdf: gpd.GeoDataFrame, world_da: xr.DataArray) -> xr.DataArray:
    print("⏳ Rasterizing AOI onto world grid …")
    # prepare shapes
    shapes = ((geom, 1) for geom in aoi_gdf.geometry)

    # grid params
    lon = world_da['lon'].values
    lat = world_da['lat'].values
    res = float(lon[1] - lon[0])
    lon_min = lon.min() - res/2
    lat_max = lat.max() + res/2
    h, w = len(lat), len(lon)
    transform = Affine.translation(lon_min, lat_max) * Affine.scale(res, -res)

    rast = features.rasterize(
        shapes=shapes,
        out_shape=(h, w),
        transform=transform,
        fill=0,
        dtype='uint8',
        all_touched=True
    )

    print("⏳ Clipping world mask to AOI …")
    clipped = np.where(rast == 1, world_da.values, np.nan)

    # subset to bounds
    mask = ~np.isnan(clipped)
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    lat_sub = lat[rows.min():rows.max()+1]
    lon_sub = lon[cols.min():cols.max()+1]
    data_sub = clipped[rows.min():rows.max()+1, cols.min():cols.max()+1]

    da_aoi = xr.DataArray(
        data_sub,
        coords={'lat': lat_sub, 'lon': lon_sub},
        dims=['lat', 'lon'],
        name='land'
    )
    da_aoi.attrs.update({
        'description': f'AOI land mask for {aoi_gdf["adm0_src"].iloc[0]} (1=land, NaN elsewhere)',
        'grid_resolution': f'{res} deg',
        'units': '1 for land, NaN otherwise'
    })
    print("✔️ AOI mask rasterized and clipped")
    return da_aoi

def save_aoi_mask(da: xr.DataArray, out_path: str):
    print(f"⏳ Saving AOI mask to {out_path} …")
    da.to_netcdf(out_path)
    print("✅ AOI land mask saved successfully")

def main():
    try:
        world_nc = os.path.join(main_dir, 'data/mask/world/wld_land_mask.nc')
        gpkg = os.path.join(main_dir, 'data/downloads/boundary/adm0_polygons.gpkg')
        iso3 = 'LKA'
        out_dir = os.path.join(main_dir, 'data/mask/iso3')
        os.makedirs(out_dir, exist_ok=True)
        out_file = os.path.join(out_dir, 'iso3_land_mask.nc')

        world_da = load_world_mask(world_nc)
        gdf_aoi = filter_aoi(gpkg, iso3)
        da_aoi = rasterize_and_clip(gdf_aoi, world_da)
        save_aoi_mask(da_aoi, out_file)

    except Exception as e:
        print(f"❌ Error: {e}")

# Run
if __name__ == "__main__":
    main()

Next, we’ll visualize the generated output.

In [ ]:
# Preview AOI land mask with Cartopy
import os
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def load_aoi_mask(path: str) -> xr.DataArray:
    print(f"⏳ Loading AOI mask from {path} …")
    da = xr.open_dataarray(path)
    print("✔️ AOI mask loaded")
    return da

def ensure_lat_ascending(da: xr.DataArray) -> xr.DataArray:
    if da.lat[0] > da.lat[-1]:
        print("⏳ Sorting latitudes ascending …")
        da = da.sortby('lat')
        print("✔️ Latitudes sorted")
    else:
        print("✔️ Latitudes already ascending")
    return da

def plot_mask(da: xr.DataArray):
    print("⏳ Plotting AOI mask …")
    lon = da.lon.values
    lat = da.lat.values
    z = da.values

    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    mesh = ax.pcolormesh(
        lon, lat, z,
        cmap='Greens',
        shading='auto',
        transform=ccrs.PlateCarree()
    )
    cbar = plt.colorbar(mesh, ax=ax, label='Land = 1')
    ax.coastlines(resolution='10m', linewidth=0.8)
    ax.gridlines(draw_labels=True)
    ax.set_title("AOI Land Mask")
    plt.show()
    print("✅ Plot complete")

def main():
    try:
        mask_path = os.path.join(main_dir, 'data/mask/iso3/iso3_land_mask.nc')
        da_mask = load_aoi_mask(mask_path)
        da_mask = ensure_lat_ascending(da_mask)
        plot_mask(da_mask)
    except Exception as e:
        print(f"❌ Error during preview: {e}")

# Run
if __name__ == "__main__":
    main()

#### 4.2.2 Bounding Box/Custom Polygon

In cases where the AOI does not neatly align with country boundaries, we can define it using a bounding box or a custom polygon. This provides more flexibility when working with specific regions that don't correspond to administrative borders.

We’ll walk through how to generate an AOI using a bounding box or a polygon defined by a GeoJSON text from tools such as [geojson.io](https://geojson.io). This method is particularly useful when analyzing non-administrative regions or when the focus area extends beyond country boundaries.

In [ ]:
# Generate and save AOI‐specific land mask from GeoJSON input
import os
import json
import geopandas as gpd
import numpy as np
import xarray as xr
from rasterio import features
from affine import Affine

def load_world_mask(path: str) -> xr.DataArray:
    print(f"⏳ Loading world land mask from {path} …")
    da = xr.open_dataarray(path)
    print("✔️ World land mask loaded")
    return da

def load_aoi_from_geojson(geojson_text: str) -> gpd.GeoDataFrame:
    print("⏳ Parsing AOI from GeoJSON input …")
    try:
        gj = json.loads(geojson_text)
        gdf = gpd.GeoDataFrame.from_features(gj["features"])
        gdf.set_crs("EPSG:4326", inplace=True)
        print(f"✔️ Loaded AOI with {len(gdf)} feature(s)")
        return gdf
    except Exception as e:
        raise RuntimeError(f"Failed to parse GeoJSON: {e}")

def rasterize_and_clip(aoi_gdf: gpd.GeoDataFrame, world_da: xr.DataArray) -> xr.DataArray:
    print("⏳ Rasterizing AOI onto world grid …")
    shapes = ((geom, 1) for geom in aoi_gdf.geometry)

    lon = world_da['lon'].values
    lat = world_da['lat'].values
    res = float(lon[1] - lon[0])
    lon_min = lon.min() - res/2
    lat_max = lat.max() + res/2
    h, w = len(lat), len(lon)
    transform = Affine.translation(lon_min, lat_max) * Affine.scale(res, -res)

    rast = features.rasterize(
        shapes,
        out_shape=(h, w),
        transform=transform,
        fill=0,
        dtype='uint8',
        all_touched=True
    )
    print("✔️ Rasterization complete")

    print("⏳ Clipping world mask to AOI …")
    clipped = np.where(rast == 1, world_da.values, np.nan)

    # subset to data bounds
    mask = ~np.isnan(clipped)
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    lat_sub = lat[rows.min():rows.max()+1]
    lon_sub = lon[cols.min():cols.max()+1]
    data_sub = clipped[rows.min():rows.max()+1, cols.min():cols.max()+1]

    da_aoi = xr.DataArray(
        data_sub,
        coords={'lat': lat_sub, 'lon': lon_sub},
        dims=['lat', 'lon'],
        name='land'
    )
    da_aoi.attrs.update({
        'description': 'AOI land mask from custom GeoJSON (1=land, NaN elsewhere)',
        'grid_resolution': f'{res} deg',
        'units': '1 for land, NaN otherwise'
    })
    print("✔️ AOI mask rasterized and clipped")
    return da_aoi

def save_aoi_mask(da: xr.DataArray, out_path: str):
    print(f"⏳ Saving AOI mask to {out_path} …")
    da.to_netcdf(out_path)
    print("✅ AOI land mask saved successfully")

def main():
    try:
        # --- User parameters ---
        geojson_input = """
        {
          "type": "FeatureCollection",
          "features": [
            {
              "type": "Feature",
              "properties": {},
              "geometry": {
                "type": "Polygon",
                "coordinates": [
                  [
                    [118.59479107734273, -9.01996206314675],
                    [118.59479107734273, -10.672698952161383],
                    [121.2472837636185, -10.672698952161383],
                    [121.2472837636185, -9.01996206314675],
                    [118.59479107734273, -9.01996206314675]
                  ]
                ]
              }
            }
          ]
        }
        """
        world_nc = os.path.join(main_dir, 'data/mask/world/wld_land_mask.nc')
        out_dir = os.path.join(main_dir, 'data/mask/aoi')
        os.makedirs(out_dir, exist_ok=True)
        out_file = os.path.join(out_dir, 'aoi_land_mask.nc')

        # --- Workflow ---
        world_da = load_world_mask(world_nc)
        aoi_gdf = load_aoi_from_geojson(geojson_input)
        da_aoi = rasterize_and_clip(aoi_gdf, world_da)
        save_aoi_mask(da_aoi, out_file)

        print("🎉 AOI land mask generation complete.")

    except Exception as e:
        print(f"❌ Error: {e}")

# Run
if __name__ == "__main__":
    main()

Next, we’ll visualize the generated output.

In [ ]:
# Preview AOI land mask with Cartopy
import os
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def load_aoi_mask(path: str) -> xr.DataArray:
    print(f"⏳ Loading AOI mask from {path} …")
    da = xr.open_dataarray(path)
    print("✔️ AOI mask loaded")
    return da

def ensure_lat_ascending(da: xr.DataArray) -> xr.DataArray:
    if da.lat[0] > da.lat[-1]:
        print("⏳ Sorting latitudes ascending …")
        da = da.sortby('lat')
        print("✔️ Latitudes sorted")
    else:
        print("✔️ Latitudes already ascending")
    return da

def plot_mask(da: xr.DataArray):
    print("⏳ Plotting AOI mask …")
    lon = da.lon.values
    lat = da.lat.values
    z = da.values

    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    mesh = ax.pcolormesh(
        lon, lat, z,
        cmap='Greens',
        shading='auto',
        transform=ccrs.PlateCarree()
    )
    cbar = plt.colorbar(mesh, ax=ax, label='Land = 1')
    ax.coastlines(resolution='10m', linewidth=0.8)
    ax.gridlines(draw_labels=True)
    ax.set_title("AOI Land Mask")
    plt.show()
    print("✅ Plot complete")

def main():
    try:
        mask_path = os.path.join(main_dir, 'data/mask/aoi/aoi_land_mask.nc')
        da_mask = load_aoi_mask(mask_path)
        da_mask = ensure_lat_ascending(da_mask)
        plot_mask(da_mask)
    except Exception as e:
        print(f"❌ Error during preview: {e}")

# Run
if __name__ == "__main__":
    main()

#### 4.2.3 User-Supplied Data

For those who have their own custom data, this option allows us to define the AOI using shapefiles, GeoJSON, or GeoPackages. This approach is especially useful for projects with non-standard administrative divisions or custom region definitions.

In this section, we describe how to load and process user-supplied vector data. Once loaded, we can rasterize the AOI to match the global land mask and proceed with further spatial analysis.

In [ ]:
# Generate and save AOI‐specific land mask from user‐provided vector
import os
import geopandas as gpd
import numpy as np
import xarray as xr
from rasterio import features
from affine import Affine

def load_world_mask(path: str) -> xr.DataArray:
    print(f"⏳ Loading world land mask from {path} …")
    da = xr.open_dataarray(path)
    print("✔️ World land mask loaded")
    return da

def load_aoi(aoi_path: str) -> gpd.GeoDataFrame:
    """
    Reads a user‐provided AOI vector. Supported formats: .shp, .gpkg, .geojson
    """
    print(f"⏳ Loading AOI from {aoi_path} …")
    gdf = gpd.read_file(aoi_path)
    print(f"✔️ Loaded {len(gdf)} feature(s) from AOI")
    return gdf

def rasterize_and_clip(aoi_gdf: gpd.GeoDataFrame, world_da: xr.DataArray) -> xr.DataArray:
    print("⏳ Rasterizing AOI onto world grid …")
    shapes = ((geom, 1) for geom in aoi_gdf.geometry)

    lon = world_da['lon'].values
    lat = world_da['lat'].values
    res = float(lon[1] - lon[0])
    lon_min = lon.min() - res/2
    lat_max = lat.max() + res/2
    h, w = len(lat), len(lon)
    transform = Affine.translation(lon_min, lat_max) * Affine.scale(res, -res)

    rast = features.rasterize(
        shapes=shapes,
        out_shape=(h, w),
        transform=transform,
        fill=0,
        dtype='uint8',
        all_touched=True
    )

    print("⏳ Clipping world mask to AOI …")
    clipped = np.where(rast == 1, world_da.values, np.nan)

    mask = ~np.isnan(clipped)
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    lat_sub = lat[rows.min():rows.max()+1]
    lon_sub = lon[cols.min():cols.max()+1]
    data_sub = clipped[rows.min():rows.max()+1, cols.min():cols.max()+1]

    da_aoi = xr.DataArray(
        data_sub,
        coords={'lat': lat_sub, 'lon': lon_sub},
        dims=['lat', 'lon'],
        name='land'
    )
    da_aoi.attrs.update({
        'description': 'AOI land mask (1=land, NaN elsewhere)',
        'grid_resolution': f'{res} deg',
        'units': '1 for land, NaN otherwise'
    })
    print("✔️ AOI mask rasterized and clipped")
    return da_aoi

def save_aoi_mask(da: xr.DataArray, out_path: str):
    print(f"⏳ Saving AOI mask to {out_path} …")
    da.to_netcdf(out_path)
    print("✅ AOI land mask saved successfully")

def main():
    try:
        world_nc = os.path.join(main_dir, 'data/mask/world/wld_land_mask.nc')

        # === USER INPUT ===
        # Place your AOI file (shapefile, gpkg, or geojson) under data/downloads/userdata/
        # e.g. aoi_filename = 'my_region.geojson'
        userdata_dir = os.path.join(main_dir, 'data/downloads/userdata')
        aoi_filename = 'java.shp'  # ← change to your file name
        aoi_path = os.path.join(userdata_dir, aoi_filename)
        # ==================

        out_dir = os.path.join(main_dir, 'data/mask/aoi')
        os.makedirs(out_dir, exist_ok=True)
        out_file = os.path.join(out_dir, 'user_land_mask.nc')

        world_da = load_world_mask(world_nc)
        aoi_gdf = load_aoi(aoi_path)
        da_aoi = rasterize_and_clip(aoi_gdf, world_da)
        save_aoi_mask(da_aoi, out_file)

    except Exception as e:
        print(f"❌ Error: {e}")

# Run
if __name__ == "__main__":
    main()

Next, we’ll visualize the generated output.

In [ ]:
# Preview AOI land mask with Cartopy
import os
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def load_aoi_mask(path: str) -> xr.DataArray:
    print(f"⏳ Loading AOI mask from {path} …")
    da = xr.open_dataarray(path)
    print("✔️ AOI mask loaded")
    return da

def ensure_lat_ascending(da: xr.DataArray) -> xr.DataArray:
    if da.lat[0] > da.lat[-1]:
        print("⏳ Sorting latitudes ascending …")
        da = da.sortby('lat')
        print("✔️ Latitudes sorted")
    else:
        print("✔️ Latitudes already ascending")
    return da

def plot_mask(da: xr.DataArray):
    print("⏳ Plotting AOI mask …")
    lon = da.lon.values
    lat = da.lat.values
    z = da.values

    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    mesh = ax.pcolormesh(
        lon, lat, z,
        cmap='Greens',
        shading='auto',
        transform=ccrs.PlateCarree()
    )
    cbar = plt.colorbar(mesh, ax=ax, label='Land = 1')
    ax.coastlines(resolution='10m', linewidth=0.8)
    ax.gridlines(draw_labels=True)
    ax.set_title("AOI Land Mask")
    plt.show()
    print("✅ Plot complete")

def main():
    try:
        mask_path = os.path.join(main_dir, 'data/mask/aoi/user_land_mask.nc')
        da_mask = load_aoi_mask(mask_path)
        da_mask = ensure_lat_ascending(da_mask)
        plot_mask(da_mask)
    except Exception as e:
        print(f"❌ Error during preview: {e}")

# Run
if __name__ == "__main__":
    main()

## 5 Summary

In this notebook we established the spatial foundation for the bias-correction workflow:

1. **Global land mask** (Section 4.1) -- We downloaded admin-0 boundaries from Fieldmaps.io and rasterized them onto a 0.1-degree grid aligned with the IMERG satellite precipitation product. The result is a global binary mask (1 = land, NaN = water) stored as NetCDF.

2. **AOI extraction** (Section 4.2) -- Using the global mask as a starting point, we demonstrated three approaches to define an Area of Interest:
   - **Country boundaries** (4.2.1) -- Filter by ISO3 code to obtain a national-level mask (example: Sri Lanka, `LKA`).
   - **Bounding box / custom polygon** (4.2.2) -- Paste a GeoJSON geometry (e.g. from [geojson.io](https://geojson.io)) to clip an arbitrary rectangular or polygonal region.
   - **User-supplied vector data** (4.2.3) -- Load a local shapefile, GeoPackage, or GeoJSON file for non-standard AOI definitions.

Each method produces a grid-aligned NetCDF land mask that is directly usable by the downstream notebooks. The mask is applied during bias correction (notebook 02), metric computation (notebook 03), and quality assessment (notebook 04) to restrict processing to land pixels within the target domain.

**Next step**: Proceed to **Notebook 01 -- Data Acquisition** to download the IMERG and CPC-UNI precipitation datasets for the AOI defined here.